# 03 - Power BI Preparation and Dashboard Control Totals

This learning notebook covers **Part 1, Task 4** before the dashboard is built in Power BI Desktop. It calculates control totals that the Power Query transformations, visuals, KPI cards and slicers must reproduce.

## Learning objectives

1. Confirm row count, source-column count, nulls and appropriate types.
2. Define Hour, Celsius temperature and Traffic Category transformations.
3. Validate daily, hourly and weather-level dashboard results.
4. Identify scatter-plot ranges and notable data-quality outliers.
5. Define KPI cards and slicers consistently.
6. Use control totals to verify the final `.pbix` file.

> Power BI Desktop requires Windows. This notebook does not create a `.pbix`; it prepares and verifies the exact logic you will reproduce on a Windows computer or approved Windows environment.

## 1. Locate the verified SQLite database

The dashboard must import the supplied CSV, but the verified database lets us calculate trustworthy comparison values before opening Power BI.

In [11]:
import math
import sqlite3
import statistics
from pathlib import Path

DATABASE_RELATIVE_PATH = Path("part1_data_analytics/sql/traffic_analysis.db")

def find_project_root(start: Path, relative_path: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / relative_path).is_file():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path}.")

PROJECT_ROOT = find_project_root(Path.cwd(), DATABASE_RELATIVE_PATH)
DATABASE_PATH = PROJECT_ROOT / DATABASE_RELATIVE_PATH
connection = sqlite3.connect(DATABASE_PATH)

print(f"Database: {DATABASE_PATH.relative_to(PROJECT_ROOT)}")

Database: part1_data_analytics/sql/traffic_analysis.db


## 2. Create a compact query-display helper

In [12]:
def run_query(sql, parameters=()):
    cursor = connection.execute(sql, parameters)
    headers = [item[0] for item in cursor.description]
    records = cursor.fetchall()
    text_rows = [["NULL" if value is None else str(value) for value in row] for row in records]
    widths = [len(header) for header in headers]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    print(" | ".join(header.ljust(width) for header, width in zip(headers, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in text_rows:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))
    print(f"\n{len(records)} row(s) returned.")
    return records

## 3. Validate source data quality and types

The CSV contains nine source columns. `row_id` exists only in SQLite and must not be counted as a CSV field. The `None` holiday text is a valid category, not a null.

In [13]:
quality_rows = run_query(
    """
    SELECT
        COUNT(*) AS source_rows,
        COUNT(DISTINCT date_time) AS distinct_hours,
        SUM(
            CASE
                WHEN holiday IS NULL OR temp IS NULL OR rain_1h IS NULL
                  OR snow_1h IS NULL OR clouds_all IS NULL
                  OR weather_main IS NULL OR weather_description IS NULL
                  OR date_time IS NULL OR traffic_volume IS NULL
                THEN 1 ELSE 0
            END
        ) AS rows_with_nulls
    FROM traffic
    """
)

source_columns = [
    "holiday", "temp", "rain_1h", "snow_1h", "clouds_all",
    "weather_main", "weather_description", "date_time",
    "traffic_volume",
]
power_query_types = {
    "holiday": "Text",
    "temp": "Decimal Number",
    "rain_1h": "Decimal Number",
    "snow_1h": "Decimal Number",
    "clouds_all": "Whole Number",
    "weather_main": "Text",
    "weather_description": "Text",
    "date_time": "Date/Time",
    "traffic_volume": "Whole Number",
}

print(f"Source-column count: {len(source_columns)}")
print("Recommended Power Query types:")
for column, data_type in power_query_types.items():
    print(f"  {column:<20} -> {data_type}")

source_rows | distinct_hours | rows_with_nulls
------------+----------------+----------------
48204       | 40575          | 0              

1 row(s) returned.
Source-column count: 9
Recommended Power Query types:
  holiday              -> Text
  temp                 -> Decimal Number
  rain_1h              -> Decimal Number
  snow_1h              -> Decimal Number
  clouds_all           -> Whole Number
  weather_main         -> Text
  weather_description  -> Text
  date_time            -> Date/Time
  traffic_volume       -> Whole Number


## 4. Define and test the required derived columns

The category boundaries must be unambiguous:

- Low: traffic volume below 4,500
- Medium: traffic volume from 4,500 through 5,500, inclusive
- High: traffic volume above 5,500

Celsius is calculated as `Kelvin - 273.15`, and Hour is extracted as an integer from 0 to 23.

In [14]:
sample_transform_rows = run_query(
    """
    SELECT
        date_time,
        CAST(strftime('%H', date_time) AS INTEGER) AS hour,
        temp AS temperature_kelvin,
        ROUND(temp - 273.15, 2) AS temperature_celsius,
        traffic_volume,
        CASE
            WHEN traffic_volume < 4500 THEN 'Low'
            WHEN traffic_volume <= 5500 THEN 'Medium'
            ELSE 'High'
        END AS traffic_category
    FROM traffic
    ORDER BY row_id
    LIMIT 10
    """,
)

category_rows = run_query(
    """
    SELECT
        CASE
            WHEN traffic_volume < 4500 THEN 'Low'
            WHEN traffic_volume <= 5500 THEN 'Medium'
            ELSE 'High'
        END AS traffic_category,
        COUNT(*) AS source_records
    FROM traffic
    GROUP BY traffic_category
    ORDER BY traffic_category
    """,
)

date_time           | hour | temperature_kelvin | temperature_celsius | traffic_volume | traffic_category
--------------------+------+--------------------+---------------------+----------------+-----------------
2012-10-02 09:00:00 | 9    | 288.28             | 15.13               | 5545           | High            
2012-10-02 10:00:00 | 10   | 289.36             | 16.21               | 4516           | Medium          
2012-10-02 11:00:00 | 11   | 289.58             | 16.43               | 4767           | Medium          
2012-10-02 12:00:00 | 12   | 290.13             | 16.98               | 5026           | Medium          
2012-10-02 13:00:00 | 13   | 291.14             | 17.99               | 4918           | Medium          
2012-10-02 14:00:00 | 14   | 291.72             | 18.57               | 5181           | Medium          
2012-10-02 15:00:00 | 15   | 293.17             | 20.02               | 5584           | High            
2012-10-02 16:00:00 | 16   | 293.86           

### Equivalent Power Query transformations

After importing and promoting the CSV headers, set the nine source types, then add these custom columns:

```powerquery
Added Hour = Table.AddColumn(
    Changed Types,
    "Hour",
    each Time.Hour(DateTime.Time([date_time])),
    Int64.Type
)

Added Temperature Celsius = Table.AddColumn(
    Added Hour,
    "Temperature_Celsius",
    each [temp] - 273.15,
    type number
)

Added Traffic Category = Table.AddColumn(
    Added Temperature Celsius,
    "Traffic_Category",
    each if [traffic_volume] < 4500 then "Low"
         else if [traffic_volume] <= 5500 then "Medium"
         else "High",
    type text
)
```

The final Power Query table should have 48,204 rows and 12 columns: nine source columns plus three derived columns.

## 5. Daily traffic trend controls for 2015–2017

The dashboard line chart should place date on the X-axis and use average traffic volume as the value. The summary below provides year-level control totals for the daily series. The unequal number of observed days must be disclosed.

In [15]:
daily_summary_rows = run_query(
    """
    WITH daily AS (
        SELECT
            CAST(strftime('%Y', date_time) AS INTEGER) AS year,
            date(date_time) AS traffic_date,
            AVG(traffic_volume) AS daily_average_traffic
        FROM traffic
        WHERE CAST(strftime('%Y', date_time) AS INTEGER) BETWEEN 2015 AND 2017
        GROUP BY year, traffic_date
    )
    SELECT
        year,
        COUNT(*) AS observed_days,
        ROUND(AVG(daily_average_traffic), 2) AS average_of_daily_averages,
        ROUND(MIN(daily_average_traffic), 2) AS lowest_daily_average,
        ROUND(MAX(daily_average_traffic), 2) AS highest_daily_average
    FROM daily
    GROUP BY year
    ORDER BY year
    """,
)

year | observed_days | average_of_daily_averages | lowest_daily_average | highest_daily_average
-----+---------------+---------------------------+----------------------+----------------------
2015 | 195           | 3232.94                   | 1478.15              | 4681.0               
2016 | 366           | 3199.3                    | 192.94               | 4238.74              
2017 | 365           | 3347.68                   | 1751.05              | 4555.17              

3 row(s) returned.


## 6. Hourly traffic controls for 2017

The column chart should contain exactly 24 hour categories. The source-record count is included as a diagnostic because repeated weather observations vary by hour.

In [16]:
hourly_2017_rows = run_query(
    """
    SELECT
        CAST(strftime('%H', date_time) AS INTEGER) AS hour,
        COUNT(*) AS source_records,
        ROUND(AVG(traffic_volume), 2) AS average_traffic_volume
    FROM traffic
    WHERE strftime('%Y', date_time) = '2017'
    GROUP BY hour
    ORDER BY hour
    """,
)

busiest_hour_2017 = max(hourly_2017_rows, key=lambda row: row[2])
quietest_hour_2017 = min(hourly_2017_rows, key=lambda row: row[2])
print(f"Busiest 2017 hour: {busiest_hour_2017[0]:02d}:00, average {busiest_hour_2017[2]:,.2f}")
print(f"Quietest 2017 hour: {quietest_hour_2017[0]:02d}:00, average {quietest_hour_2017[2]:,.2f}")

hour | source_records | average_traffic_volume
-----+----------------+-----------------------
0    | 454            | 921.45                
1    | 459            | 546.99                
2    | 453            | 398.19                
3    | 456            | 381.71                
4    | 451            | 735.82                
5    | 449            | 2172.63               
6    | 449            | 4305.28               
7    | 455            | 4879.8                
8    | 448            | 4716.23               
9    | 457            | 4488.8                
10   | 449            | 4259.88               
11   | 427            | 4546.28               
12   | 417            | 4804.14               
13   | 423            | 4794.32               
14   | 424            | 4982.5                
15   | 432            | 5316.31               
16   | 430            | 5797.54               
17   | 433            | 5469.6                
18   | 434            | 4386.57               
19   | 432   

## 7. Weather-impact controls

The dashboard should use `weather_main` for the condition categories. Detailed weather descriptions are too granular for the required high-level comparison.

In [17]:
weather_rows = run_query(
    """
    SELECT
        weather_main,
        COUNT(*) AS source_records,
        ROUND(AVG(traffic_volume), 2) AS average_traffic_volume
    FROM traffic
    GROUP BY weather_main
    ORDER BY average_traffic_volume DESC
    """,
)

highest_weather = weather_rows[0]
lowest_weather = weather_rows[-1]
weather_average_difference = highest_weather[2] - lowest_weather[2]

print(f"Highest average: {highest_weather[0]} ({highest_weather[2]:,.2f})")
print(f"Lowest average: {lowest_weather[0]} ({lowest_weather[2]:,.2f})")
print(f"Difference: {weather_average_difference:,.2f}")

weather_main | source_records | average_traffic_volume
-------------+----------------+-----------------------
Clouds       | 15164          | 3618.45               
Haze         | 1360           | 3502.1                
Rain         | 5672           | 3317.91               
Drizzle      | 1821           | 3290.73               
Smoke        | 20             | 3237.65               
Clear        | 13391          | 3055.91               
Snow         | 2876           | 3016.84               
Thunderstorm | 1034           | 3001.62               
Mist         | 5950           | 2932.96               
Fog          | 912            | 2703.72               
Squall       | 4              | 2061.75               

11 row(s) returned.
Highest average: Clouds (3,618.45)
Lowest average: Squall (2,061.75)
Difference: 1,556.70


### Weather interpretation caution

`Clouds` has the highest raw average traffic and `Squall` the lowest. However, `Squall` has only four records, so its mean is unstable and should not support a broad policy conclusion. Weather categories also occur at different hours and seasons; the visual is descriptive rather than causal.

## 8. Temperature-versus-traffic scatter controls

The scatter plot should use `Temperature_Celsius` on the X-axis and `traffic_volume` on the Y-axis. A broad cloud of points is expected because the overall correlation is very weak. We inspect the temperature distribution for high-traffic records and identify obvious outliers.

In [18]:
scatter_rows = connection.execute(
    "SELECT temp, traffic_volume FROM traffic ORDER BY row_id"
).fetchall()
valid_high_traffic_temperatures = sorted(
    temperature
    for temperature, traffic_volume in scatter_rows
    if traffic_volume > 5500 and temperature > 0
)
temperature_quartiles = statistics.quantiles(
    valid_high_traffic_temperatures,
    n=4,
    method="inclusive",
)
zero_kelvin_count = sum(temperature == 0 for temperature, _ in scatter_rows)
zero_traffic_count = sum(traffic_volume == 0 for _, traffic_volume in scatter_rows)

print(f"High-traffic records with valid temperatures: {len(valid_high_traffic_temperatures):,}")
print(f"High-traffic temperature minimum: {min(valid_high_traffic_temperatures):.2f} K")
print(f"High-traffic temperature Q1:      {temperature_quartiles[0]:.2f} K")
print(f"High-traffic temperature median:  {statistics.median(valid_high_traffic_temperatures):.2f} K")
print(f"High-traffic temperature Q3:      {temperature_quartiles[2]:.2f} K")
print(f"High-traffic temperature maximum: {max(valid_high_traffic_temperatures):.2f} K")
print(f"Zero-Kelvin outliers: {zero_kelvin_count}")
print(f"Zero-traffic observations: {zero_traffic_count}")

High-traffic records with valid temperatures: 7,100
High-traffic temperature minimum: 247.15 K
High-traffic temperature Q1:      274.31 K
High-traffic temperature median:  284.38 K
High-traffic temperature Q3:      292.35 K
High-traffic temperature maximum: 309.29 K
Zero-Kelvin outliers: 10
Zero-traffic observations: 2


## 9. KPI card controls

The phrase *total hours analysed* is ambiguous because the CSV has 48,204 rows but only 40,575 distinct hourly timestamps. Use distinct timestamps for an honest hours KPI and retain source-row count as a data-quality reference.

In [19]:
kpi_rows = run_query(
    """
    SELECT
        COUNT(*) AS source_rows,
        COUNT(DISTINCT date_time) AS total_hours_analyzed,
        ROUND(AVG(traffic_volume), 2) AS average_traffic_volume,
        ROUND(AVG(temp), 2) AS average_temperature_kelvin,
        ROUND(AVG(temp - 273.15), 2) AS average_temperature_celsius
    FROM traffic
    """,
)

source_rows | total_hours_analyzed | average_traffic_volume | average_temperature_kelvin | average_temperature_celsius
------------+----------------------+------------------------+----------------------------+----------------------------
48204       | 40575                | 3259.82                | 281.21                     | 8.06                       

1 row(s) returned.


### Suggested DAX measures

Assuming the imported table is named `Traffic`:

```dax
Total Hours Analyzed = DISTINCTCOUNT(Traffic[date_time])
Average Traffic Volume = AVERAGE(Traffic[traffic_volume])
Average Temperature Celsius = AVERAGE(Traffic[Temperature_Celsius])
```

Expected unfiltered card values are 40,575 hours, 3,259.82 vehicles and 8.06 °C. KPI cards should respond to the slicers.

## 10. Dashboard blueprint

Create one clear report page with:

1. **Top row:** Total Hours, Average Traffic and Average Temperature KPI cards.
2. **Daily trend line chart:** date on X-axis, Average Traffic Volume measure on Y-axis, filtered to 2015–2017.
3. **2017 hourly column chart:** Hour on X-axis and Average Traffic Volume on Y-axis.
4. **Weather bar chart:** `weather_main` and Average Traffic Volume, sorted descending.
5. **Scatter plot:** Temperature Celsius versus traffic volume; use weather condition as legend only if readability remains acceptable.
6. **Slicers:** Hour as a range, weather condition, and traffic category.
7. **Data note:** disclose incomplete annual coverage, repeated timestamps, ten `0 K` values and the four-record Squall category.

Use descriptive titles and label Celsius explicitly. Avoid claiming that weather causes the displayed traffic differences.

## 11. Run automated checks

In [20]:
source_rows, distinct_hours, rows_with_nulls = quality_rows[0]
assert source_rows == 48_204
assert distinct_hours == 40_575
assert rows_with_nulls == 0
assert len(source_columns) == 9
assert dict(category_rows) == {"High": 7_100, "Low": 31_237, "Medium": 9_867}
assert [row[1] for row in daily_summary_rows] == [195, 366, 365]
assert len(hourly_2017_rows) == 24
assert busiest_hour_2017[0] == 16
assert quietest_hour_2017[0] == 3
assert highest_weather[0] == "Clouds"
assert lowest_weather[0] == "Squall"
assert math.isclose(weather_average_difference, 1556.70, abs_tol=0.01)
assert zero_kelvin_count == 10
assert zero_traffic_count == 2
assert kpi_rows[0][1] == 40_575
assert math.isclose(kpi_rows[0][2], 3259.82, abs_tol=0.01)

connection.close()
print("PASS - Power BI preparation and dashboard control totals matched expected results.")

PASS - Power BI preparation and dashboard control totals matched expected results.


## 12. Power BI verification checkpoint

Before the `.pbix` is considered complete, verify that:

- Power Query shows 48,204 rows and 12 final columns.
- All nine source types and three derived types are correct.
- The category counts match this notebook.
- The 2017 chart contains all 24 hours and peaks at hour 16.
- Clouds is the highest weather average and Squall the lowest, with the sparse-category warning.
- KPI cards show the expected unfiltered values and respond to all slicers.
- The report contains a visible limitations note.

### Stop point

Run all cells and confirm the final code cell reports `PASS`. Save the notebook. We will then create the Power BI build guide, Power Query script, DAX measures and dashboard verification checklist.